# Análise de Meios de Transporte das Mulheres – Brasil (2022)

Este notebook realiza as duas partes da atividade usando o arquivo `Tabela_13_Meio_de_transporte.xlsx`.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

arquivo = 'Tabela_13_Meio_de_transporte.xlsx'


## PARTE 1 – Gráficos de pizza

Leitura da aba **Exemplo gráfico Brasil**.


In [ ]:
df = pd.read_excel(arquivo, sheet_name='Exemplo gráfico Brasil', header=1)

df.columns = ['Meio de Transporte', 'Branca', 'Preta ou Parda', 'Indígena']

df


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

cores = ['#ff9999', '#66b3ff', '#99ff99', '#ffcc99', '#c2c2f0', '#ffb3e6']

grupos = ['Branca', 'Preta ou Parda', 'Indígena']

for i, col in enumerate(grupos):
    axes[i].pie(
        df[col],
        labels=df['Meio de Transporte'],
        autopct='%1.3f%%',
        startangle=180,
        colors=cores,
        wedgeprops={'edgecolor': 'white', 'linewidth': 1.2},
    )
    axes[i].set_title(f'Cor/Raça: {col}', fontsize=12, fontweight='bold')

plt.suptitle(
    'Meios de transporte utilizados pelas mulheres, segundo cor/raça – Brasil 2022',
    fontsize=14,
    fontweight='bold',
)

plt.tight_layout()
plt.savefig('grafico_transporte.png', dpi=300)
plt.show()


In [ ]:
diferencas = df.copy()

diferencas['Branca x Preta/Parda'] = (diferencas['Branca'] - diferencas['Preta ou Parda']).abs()
diferencas['Branca x Indígena'] = (diferencas['Branca'] - diferencas['Indígena']).abs()
diferencas['Preta/Parda x Indígena'] = (diferencas['Preta ou Parda'] - diferencas['Indígena']).abs()

diferencas[['Meio de Transporte', 'Branca x Preta/Parda',
            'Branca x Indígena', 'Preta/Parda x Indígena']].round(3)


## Análise dos gráficos

- **A pé:** a diferença entre mulheres brancas e pretas/pardas é de **5,4 pontos percentuais**; entre brancas e indígenas, **18,1 p.p.**; entre pretas/pardas e indígenas, **12,7 p.p.**
- **Bicicleta:** a diferença entre brancas e pretas/pardas é de **2,0 p.p.**; entre brancas e indígenas, **2,7 p.p.**; entre pretas/pardas e indígenas, **0,7 p.p.**
- **Motocicleta ou Mototáxi:** a diferença entre brancas e pretas/pardas é de **4,3 p.p.**; entre brancas e indígenas, **5,5 p.p.**; entre pretas/pardas e indígenas, **1,2 p.p.**
- **Automóvel, táxi ou assemelhados:** a diferença entre brancas e pretas/pardas é de **21,2 p.p.**; entre brancas e indígenas, **26,6 p.p.**; entre pretas/pardas e indígenas, **5,4 p.p.**
- **Transporte coletivo:** a diferença entre brancas e pretas/pardas é de **9,4 p.p.**; entre brancas e indígenas, **2,8 p.p.**; entre pretas/pardas e indígenas, **12,2 p.p.**
- **Outros:** a diferença entre brancas e pretas/pardas é de **0,1 p.p.**; entre brancas e indígenas, **3,1 p.p.**; entre pretas/pardas e indígenas, **3,0 p.p.**

A maior diferença entre os grupos aparece no uso de **automóvel, táxi ou assemelhados**, principalmente entre mulheres brancas e indígenas (**26,6 pontos percentuais**). Já o meio com menor diferença entre pretas/pardas e indígenas é **bicicleta**, com apenas **0,7 ponto percentual**.


# PARTE 2 – Estados e municípios

Leitura da aba **BR GR UF MU**.


In [ ]:
dados = pd.read_excel(arquivo, sheet_name='BR GR UF MU', header=None, skiprows=3)

dados = dados.iloc[:, :19]
dados.columns = ['Local'] + list(range(1, 19))

dados.head()


In [ ]:
estados = [
    'Acre', 'Alagoas', 'Amapá', 'Amazonas', 'Bahia', 'Ceará',
    'Distrito Federal', 'Espírito Santo', 'Goiás', 'Maranhão',
    'Mato Grosso', 'Mato Grosso do Sul', 'Minas Gerais', 'Pará',
    'Paraíba', 'Paraná', 'Pernambuco', 'Piauí', 'Rio de Janeiro',
    'Rio Grande do Norte', 'Rio Grande do Sul', 'Rondônia', 'Roraima',
    'Santa Catarina', 'São Paulo', 'Sergipe', 'Tocantins'
]

df_estados = dados[dados['Local'].isin(estados)].copy()

df_estados[['Local']]


In [ ]:
meios = [
    'A pé',
    'Bicicleta',
    'Motocicleta ou Mototaxi',
    'Automóvel, taxi ou assemelhados',
    'Transporte coletivo',
    'Outros'
]

# Separando os dados das três categorias de raça
branca = df_estados[['Local', 1, 2, 3, 4, 5, 6]].copy()
branca.columns = ['Estado'] + meios
branca['Raça'] = 'Branca'

preta = df_estados[['Local', 7, 8, 9, 10, 11, 12]].copy()
preta.columns = ['Estado'] + meios
preta['Raça'] = 'Preta ou Parda'

indigena = df_estados[['Local', 13, 14, 15, 16, 17, 18]].copy()
indigena.columns = ['Estado'] + meios
indigena['Raça'] = 'Indígena'

df_estados_longo = pd.concat([branca, preta, indigena], ignore_index=True)

df_melt = df_estados_longo.melt(
    id_vars=['Estado', 'Raça'],
    var_name='Meio de Transporte',
    value_name='Percentual'
)

df_melt['Percentual'] = pd.to_numeric(df_melt['Percentual'], errors='coerce')

df_melt.head()


In [ ]:

df_grupo = (
    df_melt.groupby(['Estado', 'Meio de Transporte'], as_index=False)['Percentual']
    .mean()
)

coletivo = (
    df_grupo[df_grupo['Meio de Transporte'] == 'Transporte coletivo']
    .sort_values('Percentual', ascending=False)
)

coletivo.head()


## Respostas – Transporte coletivo por estado

**Estado com mais mulheres, em geral, utilizando transporte coletivo:** **Rio de Janeiro**, com média de aproximadamente **52,23%** entre as três categorias de raça.

**Estado com menos mulheres, em geral, utilizando transporte coletivo:** **Roraima**, com média de aproximadamente **8,13%** entre as três categorias de raça.


In [ ]:
cidades = dados[dados['Local'].astype(str).str.contains(r'\(', regex=True, na=False)].copy()

branca_c = cidades[['Local', 1, 2, 3, 4, 5, 6]].copy()
branca_c.columns = ['Cidade'] + meios
branca_c['Raça'] = 'Branca'

preta_c = cidades[['Local', 7, 8, 9, 10, 11, 12]].copy()
preta_c.columns = ['Cidade'] + meios
preta_c['Raça'] = 'Preta ou Parda'

indigena_c = cidades[['Local', 13, 14, 15, 16, 17, 18]].copy()
indigena_c.columns = ['Cidade'] + meios
indigena_c['Raça'] = 'Indígena'

df_cidades = pd.concat([branca_c, preta_c, indigena_c], ignore_index=True)

df_cidades = df_cidades.melt(
    id_vars=['Cidade', 'Raça'],
    var_name='Meio de Transporte',
    value_name='Percentual'
)

df_cidades['Percentual'] = pd.to_numeric(df_cidades['Percentual'], errors='coerce')

df_cidades.head()


In [ ]:

df_cidades_grupo = (
    df_cidades.groupby(['Cidade', 'Meio de Transporte'])['Percentual']
    .agg(['count', 'mean'])
    .reset_index()
)

df_cidades_grupo = df_cidades_grupo[df_cidades_grupo['count'] == 3]

bicicleta = (
    df_cidades_grupo[df_cidades_grupo['Meio de Transporte'] == 'Bicicleta']
    .sort_values('mean', ascending=False)
)

carro = (
    df_cidades_grupo[df_cidades_grupo['Meio de Transporte'] == 'Automóvel, taxi ou assemelhados']
    .sort_values('mean', ascending=False)
)

print('Bicicleta:')
print(bicicleta.head(1))

print('\nCarro:')
print(carro.head(1))


## Respostas – Municípios

**Cidade do Brasil com mais mulheres, em geral, andando de bicicleta:** **Lagoa Grande (MG)**, com média de aproximadamente **67,87%** entre as três categorias de raça.

**Cidade do Brasil com mais mulheres, em geral, utilizando carro:** **Bom Jesus (SC)**, com média de aproximadamente **74,57%** entre as três categorias de raça.

